# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [49]:
df = pd.read_csv('data/AviationData.csv')
# The data contained a character that did not match Panda loading rules so using this encoding, Python now interprets it accordingly

df.isnull().sum()
df.info()
df.describe()
df.columns.tolist()


<class 'pandas.DataFrame'>
RangeIndex: 57731 entries, 0 to 57730
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Event.Id                  57731 non-null  str    
 1   Investigation.Type        57731 non-null  str    
 2   Accident.Number           57731 non-null  str    
 3   Event.Date                57731 non-null  str    
 4   Location                  57702 non-null  str    
 5   Country                   57648 non-null  str    
 6   Latitude                  21949 non-null  object 
 7   Longitude                 21944 non-null  object 
 8   Airport.Code              34691 non-null  str    
 9   Airport.Name              36271 non-null  str    
 10  Injury.Severity           56937 non-null  str    
 11  Aircraft.damage           55811 non-null  str    
 12  Aircraft.Category         57731 non-null  str    
 13  Registration.Number       57051 non-null  str    
 14  Make             

C:\Users\amira\AppData\Local\Temp\ipykernel_101228\2915543989.py:1: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/AviationData.csv')


['Event.Id',
 'Investigation.Type',
 'Accident.Number',
 'Event.Date',
 'Location',
 'Country',
 'Latitude',
 'Longitude',
 'Airport.Code',
 'Airport.Name',
 'Injury.Severity',
 'Aircraft.damage',
 'Aircraft.Category',
 'Registration.Number',
 'Make',
 'Model',
 'Amateur.Built',
 'Number.of.Engines',
 'Engine.Type',
 'FAR.Description',
 'Purpose.of.flight',
 'Total.Fatal.Injuries',
 'Total.Serious.Injuries',
 'Total.Minor.Injuries',
 'Total.Uninjured',
 'Weather.Condition',
 'Broad.phase.of.flight',
 'Report.Status',
 'Publication.Date',
 'Total.People',
 'Fatal.Serious.Injuries',
 'Fatal_Severe_Injury_Rate',
 'Destroyed',
 'Make.Model']

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [50]:
# Professionally built airplanes with records from 1983 onwards (Event Data, Amateur Built and Aircraft Category)
# INSPECTING 
df['Amateur.Built'].value_counts(dropna=False) # I want "No"
df['Aircraft.Category'].value_counts(dropna=False) # I want "Airplane" but there are 56602 NaN so use Make/Model to help check if missing category is a plane

# Figuring our any reasonable imputations
df.loc[df['Aircraft.Category'].isna(), ['Make', 'Model', 'Aircraft.Category']].head(10)
cessna_172m = df[(df['Make'] == 'Cessna') & (df['Model'] == '172M')]
cessna_172m['Aircraft.Category'].value_counts(dropna=False) 
# There are 581 missing Cessna 172m records. 86 Airplane though. So I can now fill missing categories in Airplane without guessing
df['Make'] = df['Make'].str.upper() # E.g Cessna and CESSNA both exist so both become CESSNA
df['Model'] = df['Model'].str.upper()

# There is now 217 Airplane CESSNA 172M now

missing_CESSNA_172M= ((df['Make'] == 'CESSNA') & (df['Model'] == '172M') & (df['Aircraft.Category'].isna()))
df.loc[missing_CESSNA_172M, 'Aircraft.Category'] = 'Airplane'
# This only works for one as a tester but the issue of missing Airplane Category is still there

category_counts = df.groupby(['Make', 'Model'])['Aircraft.Category'].nunique() # There is one distinct category -> Airplane
consistent_groups = category_counts[category_counts == 1] # Which groups have consistent category
# Still need the category
category_first = df.groupby(['Make', 'Model'])['Aircraft.Category'].first() # holds their names

useful_categories = category_first.loc[consistent_groups.index] # Kind of a look up table that includes make and model and points to category
for (make, model), category in useful_categories.items():
    missing_category= ((df['Make'] == make) & (df['Model'] == model) & (df['Aircraft.Category'].isna()))
    df.loc[missing_category, 'Aircraft.Category'] = category

df['Aircraft.Category'].isna().sum()

df['Event.Date'] = pd.to_datetime(df[('Event.Date')])
df['Event.Date'].dtype

df = df[df['Event.Date'] >= '1983-01-01']
df = df[df['Amateur.Built'] == 'No']
df = df[df['Aircraft.Category'] == 'Airplane']

df.shape

(57731, 34)

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [51]:
# Fatal + Serious + Minor + Uinjured
df[['Total.Fatal.Injuries', 'Total.Serious.Injuries',  'Total.Uninjured','Total.Minor.Injuries']].isna().sum()
df['Total.People'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] + df['Total.Uninjured'] + df['Total.Minor.Injuries']
df['Fatal.Serious.Injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
(df['Total.People'] == 0).sum()
valid_total = (df['Total.People'] > 0)
Fatal_Severe_Injury_Rate = df.loc[valid_total, 'Fatal.Serious.Injuries'] / df.loc[valid_total, 'Total.People']
df['Fatal_Severe_Injury_Rate'] = Fatal_Severe_Injury_Rate
Fatal_Severe_Injury_Rate.head(10)


# The rate is (fatal + serious injuries) / total people aboard. The missing injury counts were left unknown insteadof assuming it to be zer
# The rate will remain as missing when the total is missing or zero


0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
6    1.0
7    1.0
8    0.0
9    0.0
dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [52]:
df['Aircraft.damage'].value_counts(dropna=False)
damage = df['Aircraft.damage'].map({'Destroyed': 1, 'Substantial': 0, 'Minor': 0})
df['Destroyed'] = damage
df['Destroyed'].value_counts(dropna=False)

# I treated substantial and minor as not destroyed. That does not mean they were undamaged tho
# Once again I left unknown and missing values as missing 

Destroyed
0.0    44300
1.0    11428
NaN     2003
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [53]:
df['Make'] = df['Make'].str.strip()
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50]
df = df[df['Make'].isin(valid_makes.index)]
df


# I made all manufacture names uppercase and removed extra spaces. 
# I kept makes with at least 50 records to have enough data. 

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People,Fatal.Serious.Injuries,Fatal_Severe_Injury_Rate,Destroyed,Make.Model
0,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,...,3.0,VMC,Approach,Probable Cause,NaN,4.0,0.0,0.0,0.0,CESSNA 182P
1,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,...,2.0,VMC,Landing,Probable Cause,NaN,2.0,0.0,0.0,0.0,CESSNA 182RG
2,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,...,1.0,VMC,Takeoff,Probable Cause,NaN,1.0,0.0,0.0,0.0,CESSNA 182P
3,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,NaN,NaN,CCB,CABLE,...,0.0,VMC,Approach,Probable Cause,NaN,2.0,0.0,0.0,0.0,PIPER PA-28R-200
4,20001214X41951,Accident,CHI83LA074,1983-01-01,"SPRINGBROOK, WI",United States,NaN,NaN,NaN,SPRINGBROOK,...,2.0,VMC,Landing,Probable Cause,NaN,2.0,0.0,0.0,0.0,CESSNA 140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57726,20221227106491,Accident,ERA23LA093,2022-12-26,"Annapolis, MD",United States,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,29-12-2022,1.0,1.0,1.0,NaN,PIPER PA-28-151
57727,20221227106494,Accident,ERA23LA095,2022-12-26,"Hampton, NH",United States,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,BELLANCA 7ECA
57728,20221227106497,Accident,WPR23LA075,2022-12-26,"Payson, AZ",United States,341525N,1112021W,PAN,PAYSON,...,1.0,VMC,NaN,NaN,27-12-2022,1.0,0.0,0.0,0.0,AMERICAN CHAMPION AIRCRAFT 8GCBC
57729,20221227106498,Accident,WPR23LA076,2022-12-26,"Morgan, UT",United States,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,CESSNA 210N


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [54]:
df['Model'].isna().sum()
df = df.dropna(subset='Model')
model_make_counts = df.groupby('Model')['Make'].nunique()
model_make_counts[model_make_counts > 1]

# A model alone is not unique to each make

df['Make.Model'] = df['Make'] + ' ' + df['Model'] 

df['Make.Model'].head(5)

# Some model names are used by more than one manufacturer, so I combined Make and Model to identify each aircraft type.

0         CESSNA 182P
1        CESSNA 182RG
2         CESSNA 182P
3    PIPER PA-28R-200
4          CESSNA 140
Name: Make.Model, dtype: str

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [55]:
df['Engine.Type'].value_counts(dropna=False)
df['Engine.Type'] = df['Engine.Type'].replace(['Unknown', 'UNK'], np.nan)

df['Weather.Condition'].value_counts(dropna=False)
df['Weather.Condition'] = df['Weather.Condition'].replace(['UNK', 'Unk'], np.nan)

df['Number.of.Engines'].value_counts(dropna=False)
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)
# I treated zero engine counts as unknown and replaced them with missing values, instead of guessing the number of engines.


df['Purpose.of.flight'].value_counts(dropna=False)
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace('Unknown', np.nan)
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace('Air Race show', 'Air Race/show')

df['Broad.phase.of.flight'].value_counts(dropna=False)
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace('Unknown', np.nan)




### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [56]:
df.isna().sum()
# Schedule and Air Carrier have too many NaNs
df = df.drop(columns=['Schedule', 'Air.carrier'], errors='ignore')


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [58]:
df.to_csv('data/AviationData_Cleaned.csv', index=False)